Meanshift and Camshift 
======================

Goal
----

In this chapter,

-   We will learn about the Meanshift and Camshift algorithms to track objects in videos.

Meanshift
---------

The intuition behind the meanshift is simple. Consider you have a set of points. (It can be a pixel
distribution like histogram backprojection). You are given a small window (may be a circle) and you
have to move that window to the area of maximum pixel density (or maximum number of points). It is
illustrated in the simple image given below:

![image](images/meanshift_basics.jpg)

The initial window is shown in blue circle with the name "C1". Its original center is marked in blue
rectangle, named "C1_o". But if you find the centroid of the points inside that window, you will
get the point "C1_r" (marked in small blue circle) which is the real centroid of the window. Surely
they don't match. So move your window such that the circle of the new window matches with the previous
centroid. Again find the new centroid. Most probably, it won't match. So move it again, and continue
the iterations such that the center of window and its centroid falls on the same location (or within a
small desired error). So finally what you obtain is a window with maximum pixel distribution. It is
marked with a green circle, named "C2". As you can see in the image, it has maximum number of points. The
whole process is demonstrated on a static image below:

![image](images/meanshift_face.gif)

So we normally pass the histogram backprojected image and initial target location. When the object
moves, obviously the movement is reflected in the histogram backprojected image. As a result, the meanshift
algorithm moves our window to the new location with maximum density.

### Meanshift in OpenCV

To use meanshift in OpenCV, first we need to setup the target, find its histogram so that we can
backproject the target on each frame for calculation of meanshift. We also need to provide an initial
location of window. For histogram, only Hue is considered here. Also, to avoid false values due to
low light, low light values are discarded using **cv.inRange()** function.

In [ ]:
import asyncio

import cv2 as cv
import numpy as np

video = "../videos/slow_traffic_small.mp4"
cap = cv.VideoCapture(video)

# take first frame of the video
ret, frame = cap.read()

# setup initial location of window
x, y, w, h = 300, 200, 100, 50  # simply hardcoded the values
track_window = (x, y, w, h)

# set up the ROI for tracking
roi = frame[y : y + h, x : x + w]
hsv_roi = cv.cvtColor(roi, cv.COLOR_BGR2HSV)
mask = cv.inRange(hsv_roi, np.array((0.0, 60.0, 32.0)), np.array((180.0, 255.0, 255.0)))
roi_hist = cv.calcHist([hsv_roi], [0], mask, [180], [0, 180])
cv.normalize(roi_hist, roi_hist, 0, 255, cv.NORM_MINMAX)

# Setup the termination criteria, either 10 iteration or move by at least 1 pt
term_crit = (cv.TERM_CRITERIA_EPS | cv.TERM_CRITERIA_COUNT, 10, 1)

# Rewind capture so frames are read on-demand during playback
total_frames = int(cap.get(cv.CAP_PROP_FRAME_COUNT))
cap.set(cv.CAP_PROP_POS_FRAMES, 0)

# Lazy processing: frames are tracked on-demand during playback
tracked = []  # cache of processed frames
track_windows = []  # track_window state after each processed frame


def _ensure_processed(idx):
    """Run meanshift sequentially from the last processed frame up to idx."""
    while len(tracked) <= idx:
        ret, frame = cap.read()
        if not ret:
            break
        i = len(tracked)
        tw = track_window if i == 0 else track_windows[i - 1]
        hsv = cv.cvtColor(frame, cv.COLOR_BGR2HSV)
        dst = cv.calcBackProject([hsv], [0], roi_hist, [0, 180], 1)
        ret, tw = cv.meanShift(dst, tw, term_crit)
        x, y, w, h = tw
        cv.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
        tracked.append(frame)
        track_windows.append(tw)


# --- interactive playback ---
import ipywidgets as widgets
from ipycanvas import Canvas

H, W = frame.shape[:2]
canvas = Canvas(width=W, height=H)
frame_sl = widgets.IntSlider(value=0, min=0, max=total_frames - 1, description="Frame")
play_btn = widgets.Button(description="▶ Play", button_style="success")
playing = {"val": False}
play_task = None


def _show_frame(idx):
    _ensure_processed(idx)
    canvas.put_image_data(cv.cvtColor(tracked[idx], cv.COLOR_BGR2RGB), 0, 0)


def _on_slider(change):
    if not playing["val"]:
        _show_frame(change["new"])


async def _run(start_frame):
    global play_task
    try:
        for i in range(start_frame, total_frames):
            if not playing["val"]:
                break
            frame_sl.value = i
            _show_frame(i)
            await asyncio.sleep(0.03)
    finally:
        if play_task is asyncio.current_task():
            playing["val"] = False
            play_task = None
            play_btn.description = "▶ Play"
            play_btn.button_style = "success"


def _on_play(_):
    global play_task
    if play_task is not None:
        play_task.cancel()
        return
    playing["val"] = True
    play_btn.description = "⏸ Pause"
    play_btn.button_style = "warning"
    play_task = asyncio.create_task(_run(frame_sl.value))


frame_sl.observe(_on_slider, "value")
play_btn.on_click(_on_play)
_show_frame(0)
display(widgets.HBox([frame_sl, play_btn]), canvas)

Camshift
--------

Did you closely watch the last result? There is a problem. Our window always has the same size whether
the car is very far or very close to the camera. That is not good. We need to adapt the window
size with size and rotation of the target. Once again, the solution came from "OpenCV Labs" and it
is called CAMshift (Continuously Adaptive Meanshift) published by Gary Bradsky in his paper
"Computer Vision Face Tracking for Use in a Perceptual User Interface" in 1998 @cite Bradski98 .

It applies meanshift first. Once meanshift converges, it updates the size of the window as,
\f$s = 2 \times \sqrt{\frac{M_{00}}{256}}\f$. It also calculates the orientation of the best fitting ellipse
to it. Again it applies the meanshift with new scaled search window and previous window location.
The process continues until the required accuracy is met.

![image](images/camshift_face.gif)

### Camshift in OpenCV

It is similar to meanshift, but returns a rotated rectangle (that is our result) and box
parameters (used to be passed as search window in next iteration). See the code below:

In [ ]:
cap = cv.VideoCapture(video)

# take first frame of the video
ret, frame = cap.read()

# setup initial location of window
x, y, w, h = 300, 200, 100, 50  # simply hardcoded the values
track_window = (x, y, w, h)

# set up the ROI for tracking
roi = frame[y : y + h, x : x + w]
hsv_roi = cv.cvtColor(roi, cv.COLOR_BGR2HSV)
mask = cv.inRange(hsv_roi, np.array((0.0, 60.0, 32.0)), np.array((180.0, 255.0, 255.0)))
roi_hist = cv.calcHist([hsv_roi], [0], mask, [180], [0, 180])
cv.normalize(roi_hist, roi_hist, 0, 255, cv.NORM_MINMAX)

# Setup the termination criteria, either 10 iteration or move by at least 1 pt
term_crit = (cv.TERM_CRITERIA_EPS | cv.TERM_CRITERIA_COUNT, 10, 1)

# Rewind capture so frames are read on-demand during playback
total_frames = int(cap.get(cv.CAP_PROP_FRAME_COUNT))
cap.set(cv.CAP_PROP_POS_FRAMES, 0)

# Lazy processing: frames are tracked on-demand during playback
tracked = []  # cache of processed frames
track_windows = []  # track_window state after each processed frame


def _ensure_processed(idx):
    """Run camshift sequentially from the last processed frame up to idx."""
    while len(tracked) <= idx:
        ret, frame = cap.read()
        if not ret:
            break
        i = len(tracked)
        tw = track_window if i == 0 else track_windows[i - 1]
        hsv = cv.cvtColor(frame, cv.COLOR_BGR2HSV)
        dst = cv.calcBackProject([hsv], [0], roi_hist, [0, 180], 1)
        ret, tw = cv.CamShift(dst, tw, term_crit)
        pts = cv.boxPoints(ret)
        pts = np.intp(pts)
        cv.polylines(frame, [pts], True, (0, 255, 0), 2)
        tracked.append(frame)
        track_windows.append(tw)


# --- interactive playback ---
import ipywidgets as widgets
from ipycanvas import Canvas

H, W = frame.shape[:2]
canvas = Canvas(width=W, height=H)
frame_sl = widgets.IntSlider(value=0, min=0, max=total_frames - 1, description="Frame")
play_btn = widgets.Button(description="▶ Play", button_style="success")
playing = {"val": False}


def _show_frame(idx):
    _ensure_processed(idx)
    canvas.put_image_data(cv.cvtColor(tracked[idx], cv.COLOR_BGR2RGB), 0, 0)


def _on_slider(change):
    if not playing["val"]:
        _show_frame(change["new"])


async def _run(start_frame):
    global play_task
    try:
        for i in range(start_frame, total_frames):
            if not playing["val"]:
                break
            frame_sl.value = i
            _show_frame(i)
            await asyncio.sleep(0.03)
    finally:
        if play_task is asyncio.current_task():
            playing["val"] = False
            play_task = None
            play_btn.description = "▶ Play"
            play_btn.button_style = "success"


def _on_play(_):
    global play_task
    if play_task is not None:
        play_task.cancel()
        return
    playing["val"] = True
    play_btn.description = "⏸ Pause"
    play_btn.button_style = "warning"
    play_task = asyncio.create_task(_run(frame_sl.value))


frame_sl.observe(_on_slider, "value")
play_btn.on_click(_on_play)
_show_frame(0)
display(widgets.HBox([frame_sl, play_btn]), canvas)

Additional Resources
--------------------

-  French Wikipedia page on [Camshift](http://fr.wikipedia.org/wiki/Camshift). (The two animations
    are taken from there)
2.  Bradski, G.R., "Real time face and object tracking as a component of a perceptual user
    interface," Applications of Computer Vision, 1998. WACV '98. Proceedings., Fourth IEEE Workshop
    on , vol., no., pp.214,219, 19-21 Oct 1998

Exercises
---------

-  OpenCV comes with a Python [sample](https://github.com/opencv/opencv/blob/5.x/samples/python/snippets/camshift.py) for an interactive demo of camshift. Use it, hack it, understand
    it.